In [ ]:
# Custom Vision Python client library: pip install azure-cognitiveservices-vision-customvision
from azure.cognitiveservices.vision.customvision.training import CustomVisionTrainingClient
from azure.cognitiveservices.vision.customvision.prediction import CustomVisionPredictionClient
from azure.cognitiveservices.vision.customvision.training.models import ImageFileCreateBatch, ImageFileCreateEntry, Region
from msrest.authentication import ApiKeyCredentials
from pathlib import Path
from collections import defaultdict
from dotenv import load_dotenv
from PIL import Image
from IPython.display import display
from io import BytesIO
import os, time, uuid, random

In [ ]:
load_dotenv()

# retrieve environment variables
ENDPOINT = os.getenv("VISION_TRAINING_ENDPOINT")
prediction_endpoint = os.getenv("VISION_PREDICTION_ENDPOINT")
training_key = os.getenv("VISION_TRAINING_KEY")
prediction_key = os.getenv("VISION_PREDICTION_KEY")
prediction_resource_id = os.getenv("VISION_PREDICTION_RESOURCE_ID")

missing = [k for k, v in {
    "VISION_TRAINING_ENDPOINT": ENDPOINT,
    "VISION_PREDICTION_ENDPOINT": prediction_endpoint,
    "VISION_TRAINING_KEY": training_key,
    "VISION_PREDICTION_KEY": prediction_key,
    "VISION_PREDICTION_RESOURCE_ID": prediction_resource_id
}.items() if not v]

if missing:
    raise ValueError(f"Missing environment variables: {missing}")

print("Environment variables loaded successfully.")

In [ ]:
# auth for training endpoint
credentials = ApiKeyCredentials(in_headers={"Training-key": training_key})
trainer = CustomVisionTrainingClient(ENDPOINT, credentials)
# auth for prediction endpoint
prediction_credentials = ApiKeyCredentials(in_headers={"Prediction-key": prediction_key})
predictor = CustomVisionPredictionClient(prediction_endpoint, prediction_credentials)

In [ ]:
DATASET_ROOT = Path(r"C:\Users\Edwin\Desktop\Lithan Academy\ModellingProject\datasets\Image_Classification")

TEST_IMAGE_DIR = Path(r"C:\Users\Edwin\Desktop\Lithan Academy\ModellingProject\datasets\Image_Classification\Test Data")

PROJECT_NAME = "sample-resource-name"
PUBLISH_ITERATION_NAME = "classifyModel"

CLASS_NAMES = ["Card", "Currency", "Gun", "Knife", "Phone", "Wallet"]

ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

In [ ]:
# Project Name: ABC Bank Image Classification
# Project Type: Classification
# Classification Type: Multiclass
# Domains: General

# Find the General classification domain dynamically
classification_domain = next(
    d for d in trainer.get_domains()
    if d.type == "Classification" and d.name == "General"
)

print("Using domain:", classification_domain.name, classification_domain.id)

project = trainer.create_project(
    "ABC Bank Image Classification",
    domain_id=classification_domain.id,
    classification_type="Multiclass"
)

print("Project created:")
print("Name:", project.name)
print("ID:", project.id)

In [ ]:
tag_objects = {}

for class_name in CLASS_NAMES:
    tag = trainer.create_tag(project.id, class_name)
    tag_objects[class_name] = tag
    print(f"Created tag: {class_name} -> {tag.id}")

In [ ]:
image_paths_by_tag = defaultdict(list)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_ROOT}")

for class_name in CLASS_NAMES:
    class_dir = DATASET_ROOT / class_name
    if not class_dir.exists():
        print(f"[WARNING] Missing folder: {class_dir}")
        continue

    for file_path in class_dir.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in ALLOWED_EXTENSIONS:
            image_paths_by_tag[class_name].append(file_path)

for class_name in CLASS_NAMES:
    print(f"{class_name}: {len(image_paths_by_tag[class_name])} images")

total_images = sum(len(v) for v in image_paths_by_tag.values())
print("Total images found:", total_images)

if total_images == 0:
    raise ValueError("No images found. Check your folder structure and file extensions.")

In [ ]:
def chunk_list(items, chunk_size=64):
    for i in range(0, len(items), chunk_size):
        yield items[i:i + chunk_size]

upload_failures = []

for class_name, paths in image_paths_by_tag.items():
    if not paths:
        continue

    tag_id = tag_objects[class_name].id
    print(f"\nUploading tag '{class_name}' with {len(paths)} images...")

    for batch_num, batch_paths in enumerate(chunk_list(paths, 64), start=1):
        entries = []

        for img_path in batch_paths:
            with open(img_path, "rb") as image_contents:
                entries.append(
                    ImageFileCreateEntry(
                        name=img_path.name,
                        contents=image_contents.read(),
                        tag_ids=[tag_id]
                    )
                )

        upload_result = trainer.create_images_from_files(
            project.id,
            ImageFileCreateBatch(images=entries)
        )

        if not upload_result.is_batch_successful:
            print(f"  Batch {batch_num}: partial/failed upload")
            for image_result in upload_result.images:
                if image_result.status.lower() != "ok":
                    upload_failures.append((class_name, image_result.source_url, image_result.status))
                    print(f"    Failed: {image_result.source_url} -> {image_result.status}")
        else:
            print(f"  Batch {batch_num}: uploaded {len(batch_paths)} images successfully")

print("\nUpload complete.")
print("Failures:", len(upload_failures))

In [ ]:
print("Training started...")
iteration = trainer.train_project(project.id)

while iteration.status != "Completed":
    print(f"Training status: {iteration.status}")
    time.sleep(5)
    iteration = trainer.get_iteration(project.id, iteration.id)

print("Training completed.")
print("Iteration ID:", iteration.id)
print("Iteration name:", iteration.name)
print("Iteration status:", iteration.status)

In [ ]:
trainer.publish_iteration(
    project.id,
    iteration.id,
    PUBLISH_ITERATION_NAME,
    prediction_resource_id
)

print("Iteration published successfully.")
print("Published name:", PUBLISH_ITERATION_NAME)

In [ ]:
if not TEST_IMAGE_DIR.exists():
    raise FileNotFoundError(f"Test folder not found: {TEST_IMAGE_DIR}")

ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

test_images = [
    p for p in TEST_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in ALLOWED_EXTENSIONS
]

if not test_images:
    raise ValueError(f"No test images found in: {TEST_IMAGE_DIR}")

test_image_path = random.choice(test_images)

print("Random test image selected:", test_image_path.name)

img = Image.open(test_image_path)
img.thumbnail((220, 220))   # thumbnail size
display(img)


In [ ]:
with open(test_image_path, "rb") as image_data:
    results = predictor.classify_image(
        project.id,
        PUBLISH_ITERATION_NAME,
        image_data.read()
    )

print(f"Prediction results for: {test_image_path.name}\n")

for prediction in sorted(results.predictions, key=lambda x: x.probability, reverse=True):
    print(f"{prediction.tag_name:10} : {prediction.probability:.4f}")

In [ ]:
top_prediction = max(results.predictions, key=lambda x: x.probability)

print("Top prediction")
print("--------------")
print("Label      :", top_prediction.tag_name)
print("Confidence :", f"{top_prediction.probability:.2%}")

In [ ]:
import requests
import json

image_url = "https://raw.githubusercontent.com/ismayilsiyad/Deep_Project_Video/refs/heads/main/bank-notes-941246_1280%20(1).jpg"

# Show thumbnail from URL
img_response = requests.get(image_url, timeout=30)
img_response.raise_for_status()

img = Image.open(BytesIO(img_response.content))
img.thumbnail((220, 220))

print("Test image preview:")
display(img)

# Test Using REST API
prediction_url = f"{prediction_endpoint.rstrip('/')}/customvision/v3.0/Prediction/{project.id}/classify/iterations/{PUBLISH_ITERATION_NAME}/url"

headers = {
    "Prediction-Key": prediction_key,
    "Content-Type": "application/json"
}

payload = {
    "url": image_url
}

response = requests.post(prediction_url, headers=headers, json=payload, timeout=60)
print("Status:", response.status_code)
response.raise_for_status()

result = response.json()
print(json.dumps(result, indent=2))

In [ ]:
with open(test_image_path, "rb") as image_data:
    results = predictor.classify_image(
        project.id,
        PUBLISH_ITERATION_NAME,
        image_data.read()
    )

print(f"Prediction results for: {test_image_path.name}\n")

for prediction in sorted(results.predictions, key=lambda x: x.probability, reverse=True):
    print(f"{prediction.tag_name:10} : {prediction.probability:.4f}")

In [ ]:
top_prediction = max(results.predictions, key=lambda x: x.probability)

print("Top prediction")
print("--------------")
print("Label      :", top_prediction.tag_name)
print("Confidence :", f"{top_prediction.probability:.2%}")